# 5. Gün: Gerçekleşen Oynaklık & Büyük Boyut Kovaryans
## HAR-RV, HEAVY, GARCH-X, POET & Ledoit-Wolf
### EYS'26 — Pamukkale Üniversitesi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
import statsmodels.api as sm

import sys
sys.path.insert(0, str(Path('..').resolve()))
from realized_volatility import (
    estimate_har_rv, estimate_har_rv_j, estimate_har_rv_cj,
    estimate_heavy, estimate_garch_x,
    poet_covariance, ledoit_wolf_covariance,
    marchenko_pastur_threshold,
)

plt.style.use('dark_background')
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})
COLORS = ['#a78bfa','#34d399','#f472b6','#60a5fa','#fbbf24','#fb923c']

DATA_PATH = Path('..') / 'data' / 'sample_returns.csv'
df = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True)
asset_cols = [c for c in df.columns if not c.endswith('_RV') and not c.endswith('_BPV')]
print(f"Varlık sayısı: {len(asset_cols)}, Gözlem: {len(df)}")

In [ ]:
def get_rv_bpv(df, asset):
    """Gerçekleşen varyans ve bipower varyasyon serilerini döndürür."""
    if f"{asset}_RV" in df.columns:
        rv = df[f"{asset}_RV"].dropna()
    else:
        rv = df[asset].pow(2).rolling(5).mean().dropna()
    if f"{asset}_BPV" in df.columns:
        bpv = df[f"{asset}_BPV"].dropna()
    else:
        r = df[asset]
        bpv = (np.pi / 2 * r.abs() * r.shift(1).abs()).rolling(5).mean().dropna()
    common = rv.index.intersection(bpv.index)
    return rv.loc[common], bpv.loc[common]

ASSET = asset_cols[0]
rv, bpv = get_rv_bpv(df, ASSET)
print(f"Seçili varlık: {ASSET}")
print(f"RV gözlemleri: {len(rv)}, BPV gözlemleri: {len(bpv)}")

## 1. Volatilite İmza Grafiği

Örnekleme frekansı yükseldikçe mikroyapı gürültüsü artar ve RV pozitif yanlı olur:
$$RV(f) = IV + \frac{2\eta^2}{f}$$
5 dakika kuralı: Bandi & Russell (2006) optimum frekans.

In [ ]:
freq_list = [1, 2, 3, 5, 10, 15, 20, 30, 60, 120]
IV = 0.0001
eta2_scale = 0.15
eta2 = IV * eta2_scale

ann_vols = []
for f in freq_list:
    mean_rv = IV + 2.0 * eta2 / f
    ann_vols.append(np.sqrt(252.0 * mean_rv) * 100)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(freq_list, ann_vols, lw=2.5, marker='o', ms=7, color=COLORS[0], label='Gözlemlenen RV')
ax.axhline(np.sqrt(252.0 * IV) * 100, ls='--', color='#34d399', lw=1.5,
           label=f'Gerçek IV = {np.sqrt(252.0*IV)*100:.2f}%')
ax.axvline(5, ls='--', color='#fbbf24', lw=1.5, label='5 dk optimal')
ax.set_title(f'Volatilite İmza Grafiği (η²/IV = {eta2_scale})')
ax.set_xlabel('Örnekleme Aralığı (dakika)')
ax.set_ylabel('Yıllık. Oynaklık (%)')
ax.legend(); plt.tight_layout(); plt.show()
print("Doğrusal azalış → gürültü-sinyal ayrıştırması doğrulanır (RV = IV + 2η²/f)")

## 2. HAR-RV Modeli (Corsi 2009)

$$RV_t^{(d)} = \beta_0 + \beta_d RV_{t-1}^{(d)} + \beta_w RV_{t-1}^{(w)} + \beta_m RV_{t-1}^{(m)} + \varepsilon_t$$

Günlük ($d$), haftalık ($w=$ 5 günlük ort.) ve aylık ($m=$ 22 günlük ort.) gecikmeler ile uzun bellek yaklaşık yakalanır.

In [ ]:
res_har, dfm_har = estimate_har_rv(rv)
print("=== HAR-RV Sonuçları ===")
print(f"R²        : {res_har.rsquared:.4f}")
print(f"Düz. R²   : {res_har.rsquared_adj:.4f}")
print(f"Gözlem    : {int(res_har.nobs)}")
print("\nKatsayılar:")
for name, coef, pval in zip(res_har.params.index, res_har.params, res_har.pvalues):
    sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else ''))
    print(f"  {name:20s}: {coef:.6f}  (p={pval:.4f}) {sig}")

In [ ]:
res_harj, dfm_harj = estimate_har_rv_j(rv, bpv)
res_harcj, dfm_harcj = estimate_har_rv_cj(rv, bpv)

print(f"{'Model':20s} {'R²':>8} {'Düz.R²':>8} {'N':>6}")
print('-' * 50)
for name, r in [('HAR-RV', res_har), ('HAR-RV-J', res_harj), ('HAR-RV-CJ', res_harcj)]:
    print(f"{name:20s} {r.rsquared:8.4f} {r.rsquared_adj:8.4f} {int(r.nobs):6d}")

In [ ]:
fitted = dfm_har.get('fitted', res_har.fittedvalues)
actual = dfm_har.get('RV', rv.reindex(dfm_har.index))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(actual.index, actual.values, lw=0.8, color='#60a5fa', alpha=0.7, label='Gerçek RV')
ax1.plot(fitted.index, fitted.values, lw=1.8, color=COLORS[0], label='HAR-RV Fitted')
ax1.set_title(f'Gerçek RV vs HAR-RV Fitted — {ASSET}')
ax1.set_ylabel('Gerçekleşen Varyans'); ax1.legend()

resid = dfm_har.get('resid', res_har.resid)
acf_vals = sm.tsa.acf(resid.dropna().values, nlags=20, fft=True)
conf = 1.96 / np.sqrt(len(resid.dropna()))
lags = list(range(1, 21))
clrs = ['#f87171' if abs(v) > conf else '#60a5fa' for v in acf_vals[1:21]]
ax2.bar(lags, acf_vals[1:21], color=clrs, label='ACF')
ax2.axhline(conf, ls='--', color='#fbbf24', lw=1)
ax2.axhline(-conf, ls='--', color='#fbbf24', lw=1)
ax2.set_title(f'Kalıntı ACF (±{conf:.3f} = 95% bant)'); ax2.set_xlabel('Gecikme')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y') if hasattr(ax1.xaxis, 'set_major_formatter') else None)
plt.tight_layout(); plt.show()

## 3. HEAVY & GARCH-X Modelleri

**HEAVY (Shephard & Sheppard 2010):**
$$\sigma_t^2 = \omega + \alpha\cdot RV_{t-1} + \beta\cdot\sigma_{t-1}^2$$

**GARCH-X:**
$$\sigma_t^2 = \omega + \alpha\varepsilon_{t-1}^2 + \beta\sigma_{t-1}^2 + \gamma\cdot RV_{t-1}$$

$\gamma > 0$ ve anlamlıysa RV, getiri karesinin ötesinde ek bilgi taşır.

In [ ]:
ret = df[ASSET].reindex(rv.index).dropna()
rv_aligned = rv.reindex(ret.index).dropna()
common_idx = ret.index.intersection(rv_aligned.index)
ret_c = ret.loc[common_idx]; rv_c = rv_aligned.loc[common_idx]

try:
    res_heavy = estimate_heavy(rv_c, ret_c)
    p_h = res_heavy.get('params', {})
    print(f"HEAVY  — alpha={p_h.get('alpha',0):.4f}, beta={p_h.get('beta',0):.4f}, "
          f"persist={res_heavy.get('persistence',0):.4f}")
except Exception as e:
    print(f"HEAVY hatası: {e}")
    res_heavy = None

try:
    res_gx = estimate_garch_x(ret_c, rv_c)
    p_gx = res_gx.get('params', {})
    gamma_t = res_gx.get('gamma_tstat', float('nan'))
    sig_str = f"ANLAMLI (t={gamma_t:.2f})" if abs(gamma_t) > 1.96 else f"anlamsız (t={gamma_t:.2f})"
    print(f"GARCH-X — alpha={p_gx.get('alpha',0):.4f}, beta={p_gx.get('beta',0):.4f}, "
          f"gamma={p_gx.get('gamma',0):.5f} → {sig_str}")
except Exception as e:
    print(f"GARCH-X hatası: {e}")
    res_gx = None

In [ ]:
from arch import arch_model
ret_scaled = df[ASSET].dropna() * 100
m_garch = arch_model(ret_scaled, vol='Garch', p=1, q=1, dist='normal')
r_garch = m_garch.fit(disp='off')
s2_garch = (r_garch.conditional_volatility.values / 100) ** 2
idx_garch = r_garch.conditional_volatility.index

fig, ax = plt.subplots(figsize=(12, 4))
ann = np.sqrt(252) * 100
ax.plot(idx_garch, np.sqrt(np.maximum(s2_garch, 0)) * ann, lw=1.2,
        color='#fbbf24', label='GARCH(1,1)', alpha=0.85)
if res_heavy is not None:
    s2_h = res_heavy['sigma2_series']
    ax.plot(s2_h.index, np.sqrt(np.maximum(s2_h.values, 0)) * ann,
            lw=1.4, color=COLORS[0], label='HEAVY')
if res_gx is not None:
    s2_gx = res_gx['sigma2_series']
    ax.plot(s2_gx.index, np.sqrt(np.maximum(s2_gx.values, 0)) * ann,
            lw=1.4, color=COLORS[1], label='GARCH-X')
ann_rv_ref = np.sqrt(rv * 252) * 100
ax.plot(ann_rv_ref.index, ann_rv_ref.values, lw=0.7, color='#94a3b8', alpha=0.5, label='sqrt(RV·252)')
ax.set_title(f'Koşullu Oynaklık Karşılaştırması — {ASSET} (yıllık %)')
ax.set_ylabel('Oynaklık (%)'); ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()

## 4. Büyük Boyut Kovaryans: POET & Ledoit-Wolf

**Marchenko-Pastur sınırı:** $\lambda_+ = \sigma^2(1+\sqrt{N/T})^2$ — bu sınırın üzerindeki öz değerler sinyal taşır.

**POET (Fan, Liao & Mincheva 2013):**
1. PCA: ilk $k$ öz vektör + öz değer (faktör bileşeni)
2. İdiyosenkratik kovaryansı yumuşak eşikleme ile seyrekleştir
3. $N > T$ durumunda dahi tutarlı

**Ledoit-Wolf (2004):** $\hat{\Sigma}_{LW} = (1-\delta)\hat{S} + \delta\hat{\mu}I_N$

In [ ]:
N_ASSETS = min(5, len(asset_cols))
K_FACTORS = min(3, N_ASSETS - 1)
THRESHOLD = 0.10

cols_sub = asset_cols[:N_ASSETS]
returns_sub = df[cols_sub].dropna()
T_obs = len(returns_sub)
kappa = N_ASSETS / T_obs

print(f"N={N_ASSETS}, T={T_obs}, κ=N/T={kappa:.4f}")

sample_cov = returns_sub.cov().values
poet_cov   = poet_covariance(returns_sub, k_factors=K_FACTORS, threshold=THRESHOLD)
lw_cov, delta = ledoit_wolf_covariance(returns_sub)
print(f"LW büzülme δ = {delta:.4f}")

# Marchenko-Pastur
corr_mat = returns_sub.corr().values
eigs_corr = np.sort(np.linalg.eigvalsh(corr_mat))[::-1]
mp_info = marchenko_pastur_threshold(N_ASSETS, T_obs, sigma2=1.0, eigenvalues=eigs_corr)
n_signal = mp_info['n_signal_eigenvalues'] or 0
print(f"MP λ+ = {mp_info['lambda_plus']:.4f}  →  {n_signal} sinyal öz değer, {N_ASSETS-n_signal} gürültü")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Marchenko-Pastur spectrum
clrs_eig = ['#a78bfa' if v > mp_info['lambda_plus'] else '#6b7280' for v in eigs_corr]
ax1.scatter(range(1, N_ASSETS+1), eigs_corr, c=clrs_eig, s=80, zorder=3)
ax1.axhline(mp_info['lambda_plus'], ls='--', color='#fbbf24', lw=1.8,
            label=f"λ+ = {mp_info['lambda_plus']:.3f}")
ax1.axhline(mp_info['lambda_minus'], ls=':', color='#94a3b8', lw=1.2,
            label=f"λ- = {mp_info['lambda_minus']:.3f}")
ax1.set_title(f'Öz Değer Spektrumu (mor=sinyal, gri=gürültü)')
ax1.set_xlabel('Öz Değer Sırası'); ax1.set_ylabel('Büyüklük'); ax1.legend()

# LW shrinkage
eigs_s = np.sort(np.linalg.eigvalsh(sample_cov))[::-1]
eigs_lw = np.sort(np.linalg.eigvalsh(lw_cov))[::-1]
x = list(range(1, N_ASSETS+1))
ax2.bar([xi - 0.2 for xi in x], eigs_s, 0.4, color='#fbbf24', alpha=0.75, label='Örnek')
ax2.bar([xi + 0.2 for xi in x], eigs_lw, 0.4, color=COLORS[0], alpha=0.85, label='Ledoit-Wolf')
ax2.set_title(f'Öz Değer Büzülmesi (δ={delta:.4f})')
ax2.set_xlabel('Öz Değer Sırası'); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
def cov2corr(cov):
    d = np.sqrt(np.diag(cov))
    d_inv = np.where(d > 1e-12, 1/d, 0)
    return np.outer(d_inv, d_inv) * cov

import matplotlib as mpl
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, mat, title in zip(axes,
                           [cov2corr(sample_cov), cov2corr(poet_cov), cov2corr(lw_cov)],
                           ['Örnek Kovaryans', f'POET (k={K_FACTORS})', f'Ledoit-Wolf (δ={delta:.3f})']):
    im = ax.imshow(np.round(mat, 4), cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(N_ASSETS)); ax.set_yticks(range(N_ASSETS))
    ax.set_xticklabels(cols_sub, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(cols_sub, fontsize=8)
    ax.set_title(title)
plt.colorbar(im, ax=axes[-1], fraction=0.046)
plt.suptitle('Korelasyon Matrisi Karşılaştırması', y=1.02)
plt.tight_layout(); plt.show()

## 5. MVP: POET vs Örnek Kovaryans Karşılaştırması

$$w = \frac{\hat{\Sigma}^{-1}\mathbf{1}}{\mathbf{1}^\top \hat{\Sigma}^{-1}\mathbf{1}}$$

In [ ]:
ones = np.ones(N_ASSETS)
reg = 1e-8 * np.eye(N_ASSETS)
rets_np = returns_sub.values

port_results = {}
for pname, cov_m in [('Eşit Ağırlık (1/N)', None),
                      ('Örnek-MVP', sample_cov),
                      ('POET-MVP', poet_cov),
                      ('LW-MVP', lw_cov)]:
    if cov_m is None:
        w = np.full(N_ASSETS, 1/N_ASSETS)
    else:
        try:
            H_inv = np.linalg.inv(cov_m + reg)
            w = H_inv @ ones / (ones @ H_inv @ ones)
        except np.linalg.LinAlgError:
            w = np.full(N_ASSETS, 1/N_ASSETS)
    pr = rets_np @ w
    ann_vol = float(np.std(pr) * np.sqrt(252) * 100)
    sharpe = float(np.mean(pr) / np.std(pr) * np.sqrt(252)) if np.std(pr) > 0 else np.nan
    cum = np.cumprod(1 + pr)
    roll_max = np.maximum.accumulate(cum)
    max_dd = float(((cum - roll_max) / np.where(roll_max > 0, roll_max, 1)).min() * 100)
    port_results[pname] = {'w': w, 'returns': pr, 'vol': ann_vol, 'sharpe': sharpe, 'max_dd': max_dd}

print(f"{'Portföy':30s} {'Vol%':>8} {'Sharpe':>8} {'MaxDD%':>8}")
print('-' * 60)
for pname, r in port_results.items():
    print(f"{pname:30s} {r['vol']:8.2f} {r['sharpe']:8.3f} {r['max_dd']:8.2f}")

# Cumulative return plot
fig, ax = plt.subplots(figsize=(12, 4))
idx_ret = returns_sub.index
clrs_p = ['#94a3b8', '#fbbf24', COLORS[0], COLORS[1]]
for (pname, r), c in zip(port_results.items(), clrs_p):
    cum = np.cumprod(1 + r['returns'])
    ax.plot(idx_ret, cum, lw=1.8, color=c, label=pname)
ax.set_title('Kümülatif Getiri: MVP Strateji Karşılaştırması')
ax.set_ylabel('Kümülatif Getiri'); ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()

## Özet

| Yöntem | N/T | Parametre | Avantaj |
|--------|-----|-----------|----------|
| Örnek Kovaryans | < 0.10 | N(N+1)/2 | Temel referans |
| Ledoit-Wolf | 0.10–1.00 | 1 (δ) | Analitik optimal büzülme |
| POET | > 0.50 | k + eşik | Faktör yapısı + seyreklik |

**Kaynaklar:**
- Corsi, F. (2009). A simple approximate long-memory model for realized volatility. *JFEC*, 7(2), 174–196.
- Shephard, N., & Sheppard, K. (2010). Realising the future. *J. Econometrics*, 160(1), 200–219.
- Fan, J., Liao, Y., & Mincheva, M. (2013). Large covariance estimation by thresholding principal orthogonal complements. *JRSS-B*, 75(4), 603–680.
- Ledoit, O., & Wolf, M. (2004). A well-conditioned estimator for large-dimensional covariance matrices. *J. Multivar. Anal.*, 88(2), 365–411.
- Marchenko, V. A., & Pastur, L. A. (1967). Distribution of eigenvalues for some sets of random matrices. *Mat. Sb.*, 72(4), 507–536.